In [ ]:
TARGET_VITALS = [
    "HR",
    "RESP",
    "SpO2",
]
CONDITIONING_VITALS = [
    'ABP Sys',
    'ABP Dias',
    'NBP Sys',
    'NBP Dias',
    'PVC Rate per Minute',
]

In [ ]:
!pip install wfdb --no-deps
import wfdb
import numpy as np
import pandas as pd
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 2.5 MB/s eta 0:00:00


In [ ]:
top_dir = np.arange(0, 10)
sub_dir = np.arange(0, 10000)

np.random.seed(42)
random_patient_sample = np.array([
    np.random.choice(sub_dir, size=10, replace=False)
    for _ in top_dir
])

patients = 0
missing_vitals = {v: 0 for v in TARGET_VITALS}
record_hit_rate = {"total": 0, "keep": 0, "missing": 0, "short": 0}
df = None

for t in top_dir:
    print(f"Exploring folder p0{t}...")
    sub_dir = random_patient_sample[t]
    for s in sub_dir:
        s = str(s).zfill(4)
        try:
            patient_records = wfdb.get_record_list(f"mimic3wdb-matched/1.0/p0{t}/p0{t}{s}")
        except:
            continue

        print(f"     p0{t}{s}: Ingesting numerics")
        numerics_records = [r for r in patient_records if r.endswith("n")]
        patients += 1

        for record_name in numerics_records:
            record_hit_rate["total"] += 1
            header = wfdb.rdheader(
                record_name,
                pn_dir=f"mimic3wdb-matched/1.0/p0{t}/p0{t}{s}"
            )

            if header.sig_name is None or not all(v in header.sig_name for v in TARGET_VITALS):
                record_hit_rate["missing"] += 1
                for vital in TARGET_VITALS:
                    if vital not in (header.sig_name or []):
                        missing_vitals[vital] += 1
                print("       Skipping: Missing vitals...")
                continue

            # Only download up to 24 hours of data (+10 minutes for buffer)
            max_samples = min(60 * (1440 + 10), header.sig_len)

            t0 = time.time()
            record = wfdb.rdrecord(
                record_name,
                pn_dir=f"mimic3wdb-matched/1.0/p0{t}/p0{t}{s}",
                channel_names=TARGET_VITALS+CONDITIONING_VITALS,
                physical=True,
                sampfrom=0,
                sampto=max_samples
            )
            print(f"       Downloaded in {time.time() - t0:.1f} seconds")

            rec_df = pd.DataFrame(record.p_signal, columns=record.sig_name)
            rec_df = rec_df.reindex(columns=TARGET_VITALS+CONDITIONING_VITALS)

            fs = record.fs
            samples_per_minute = round(fs * 60)
            rec_df["Minute"] = np.arange(len(rec_df)) // samples_per_minute

            # Remove first and last 5 minutes of record
            max_minute = rec_df["Minute"].max()
            rec_df = rec_df[(rec_df["Minute"] >= 5) & (rec_df["Minute"] <= max_minute - 5)]

            # Downsample to per minute if needed
            if samples_per_minute > 1:
                rec_df = rec_df.groupby("Minute")[TARGET_VITALS + CONDITIONING_VITALS].mean().reset_index()

            # Min: 2 hours of data per record
            if rec_df["Minute"].nunique() < 120:
                print(f"         Skipping: less than 2 hours of data")
                record_hit_rate["short"] += 1
                continue

            # Max: first 24 hours of data per record
            rec_df = rec_df[rec_df["Minute"] < (rec_df["Minute"].min() + 1440)]

            rec_df["Timestamp"] = pd.to_timedelta(rec_df["Minute"], unit="min")

            rec_df = rec_df.dropna(how='all')
            rec_df["Patient"] = f"p0{t}{s}"
            rec_df["Record"] = record_name
            record_hit_rate["keep"] += 1

            print(f"         {len(rec_df)} rows")

            if df is None:
                df = rec_df.dropna(how='all').reset_index(drop=True)
            else:
                df = pd.concat([df, rec_df.dropna(how='all').reset_index(drop=True)])

Exploring folder p00...
     p004742: Ingesting numerics
       Downloaded in 0.1 seconds
         128 rows
       Downloaded in 0.3 seconds
         1160 rows
       Downloaded in 0.3 seconds
         944 rows
     p000439: Ingesting numerics
       Downloaded in 0.2 seconds
         Skipping: less than 2 hours of data
       Downloaded in 0.3 seconds
         865 rows
       Downloaded in 1.2 seconds
         1440 rows
Exploring folder p01...
     p010928: Ingesting numerics
       Downloaded in 0.8 seconds
         1440 rows
Exploring folder p02...
Exploring folder p03...
Exploring folder p04...
Exploring folder p05...
     p053876: Ingesting numerics
       Downloaded in 10.2 seconds
         1374 rows
       Skipping: Missing vitals...
     p055616: Ingesting numerics
       Skipping: Missing vitals...
     p051728: Ingesting numerics
       Downloaded in 4.7 seconds
         451 rows
       Downloaded in 10.9 seconds
         1440 rows
     p059845: Ingesting numerics
       Skip

In [ ]:
patients

9

In [ ]:
record_hit_rate

{'total': 20, 'keep': 12, 'missing': 7, 'short': 1}

In [ ]:
missing_vitals

{'HR': 0, 'RESP': 0, 'SpO2': 7}

In [ ]:
print("Percent NaN values:")
print(df.isna().mean().mul(100).round(2).astype(str) + '%')

Percent NaN values:
HR                      1.14%
RESP                    2.36%
SpO2                    2.14%
ABP Sys                89.38%
ABP Dias               89.38%
NBP Sys                60.28%
NBP Dias               60.28%
PVC Rate per Minute     45.2%
Minute                   0.0%
Timestamp                0.0%
Patient                  0.0%
Record                   0.0%
dtype: object


In [ ]:
import os
os.makedirs("./data", exist_ok=True)

df.reindex(columns=["Patient", "Record", "Minute"] + TARGET_VITALS + CONDITIONING_VITALS).reset_index(drop=True)

output_path = "./data/mimic_vitals.parquet"
df.to_parquet(output_path, index=False, engine="pyarrow", compression="snappy")
print(f"Saved {len(df):,} rows to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1e6:.1f} MB")

# Print record hit rate summary
print(f"\nRecord hit rate:")
for k, v in record_hit_rate.items():
    print(f"  {k}: {v}")

Saved 13,562 rows to ./data/mimic_vitals.parquet
File size: 0.1 MB

Record hit rate:
  total: 20
  keep: 12
  missing: 7
  short: 1


In [ ]:
df.reindex(columns=["Patient", "Record", "Minute"] + TARGET_VITALS + CONDITIONING_VITALS).reset_index(drop=True)

,Patient,Record,Minute,HR,RESP,SpO2,ABP Sys,ABP Dias,NBP Sys,NBP Dias,PVC Rate per Minute
0,p004742,p004742-2166-12-29-13-04n,5,93.500000,13.400000,98.900000,NaN,NaN,NaN,NaN,NaN
1,p004742,p004742-2166-12-29-13-04n,6,92.500000,11.400000,90.600000,NaN,NaN,NaN,NaN,NaN
2,p004742,p004742-2166-12-29-13-04n,7,86.800000,11.400000,88.900000,NaN,NaN,NaN,NaN,NaN
3,p004742,p004742-2166-12-29-13-04n,8,80.500000,11.100000,99.100000,NaN,NaN,NaN,NaN,NaN
4,p004742,p004742-2166-12-29-13-04n,9,78.200000,11.000000,100.000000,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
13557,p062833,p062833-2122-01-05-15-24n,1440,78.050000,15.616667,94.483333,105.700000,54.550000,NaN,NaN,0.0
13558,p062833,p062833-2122-01-05-15-24n,1441,77.550000,22.283333,94.983333,106.900000,54.566667,NaN,NaN,0.0
13559,p062833,p062833-2122-01-05-15-24n,1442,77.983333,21.866667,94.716667,107.316667,57.316667,NaN,NaN,0.0
13560,p062833,p062833-2122-01-05-15-24n,1443,77.316667,17.283333,94.816667,108.650000,58.400000,NaN,NaN,0.0


In [ ]:
tdf = pd.read_parquet("./data/mimic_vitals.parquet")

## MIMIC-III Record Exploration

In [ ]:
record_name = "p044083-2112-05-04-19-50n"

record = wfdb.rdrecord(
    record_name,
    pn_dir="mimic3wdb-matched/1.0/p04/p044083",
    physical=True
)

In [ ]:
record.sig_name

['HR',
 'PULSE',
 'ABP',
 'ABP Sys',
 'ABP Dias',
 'ABP Mean',
 'NBP',
 'NBP Sys',
 'NBP Dias',
 'NBP Mean',
 'RESP',
 'ST III',
 'ST V',
 'SpO2',
 'PVC Rate per Minute',
 'Rhythm Status',
 'Ectopic Status',
 'Ectopic Count']

In [ ]:
record.sig_name == TARGET_VITALS

True

In [ ]:
import pandas as pd
import numpy as np

tdf = pd.DataFrame(record.p_signal, columns=record.sig_name)
fs = record.fs
tdf.index = pd.to_timedelta(np.arange(len(tdf)) / fs, unit="s")

In [ ]:
tdf

,HR,PULSE,ABP,ABP Sys,ABP Dias,ABP Mean,NBP,NBP Sys,NBP Dias,NBP Mean,RESP,ST III,ST V,SpO2,PVC Rate per Minute,Rhythm Status,Ectopic Status,Ectopic Count
0 days 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0 days 00:00:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0 days 00:00:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0 days 00:00:03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0 days 00:00:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1 days 21:48:34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1 days 21:48:35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1 days 21:48:36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1 days 21:48:37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
tdf_filtered = tdf.dropna(how='all')
tdf_filtered

,HR,PULSE,ABP,ABP Sys,ABP Dias,ABP Mean,NBP,NBP Sys,NBP Dias,NBP Mean,RESP,ST III,ST V,SpO2,PVC Rate per Minute,Rhythm Status,Ectopic Status,Ectopic Count
0 days 00:00:58,NaN,79.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
0 days 00:00:59,NaN,79.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
0 days 00:01:00,NaN,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
0 days 00:01:01,NaN,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
0 days 00:01:02,NaN,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1 days 21:11:25,NaN,86.0,NaN,NaN,NaN,NaN,NaN,111.0,64.0,75.0,52.0,NaN,NaN,99.0,5.0,63021.0,63062.0,NaN
1 days 21:11:26,94.0,86.0,NaN,NaN,NaN,NaN,NaN,111.0,64.0,75.0,49.0,NaN,NaN,99.0,5.0,63021.0,63062.0,NaN
1 days 21:11:27,93.0,86.0,NaN,NaN,NaN,NaN,NaN,111.0,64.0,75.0,49.0,NaN,NaN,99.0,5.0,63021.0,63062.0,NaN
1 days 21:11:28,93.0,86.0,NaN,NaN,NaN,NaN,NaN,111.0,64.0,75.0,47.0,NaN,NaN,99.0,6.0,63021.0,63062.0,NaN


In [ ]:
tdf_filtered[tdf_filtered["Rhythm Status"].notna()]["Rhythm Status"]

,Rhythm Status
0 days 00:01:11,63005.0
0 days 00:01:12,63005.0
0 days 00:01:13,63005.0
0 days 00:01:14,63005.0
0 days 00:01:15,63005.0
...,...
1 days 21:11:25,63021.0
1 days 21:11:26,63021.0
1 days 21:11:27,63021.0
1 days 21:11:28,63021.0


In [ ]:
record.sig_name

['HR',
 'PULSE',
 'ABP',
 'ABP Sys',
 'ABP Dias',
 'ABP Mean',
 'NBP',
 'NBP Sys',
 'NBP Dias',
 'NBP Mean',
 'RESP',
 'ST III',
 'ST V',
 'SpO2',
 'PVC Rate per Minute',
 'Rhythm Status',
 'Ectopic Status',
 'Ectopic Count']

In [ ]:
record.units

['bpm',
 'bpm',
 'mmHg',
 'mmHg',
 'mmHg',
 'mmHg',
 'mmHg',
 'mmHg',
 'mmHg',
 'mmHg',
 'pm',
 'uV',
 'uV',
 '%',
 'pm',
 '?',
 '?',
 '?']